# StyleMatch All-Source Robust Evaluation

Merges all locally available source text, rebuilds source-heldout splits, and runs content-masked style baselines.

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')
REPO = Path('/content/drive/MyDrive/style_matching')
RUNTIME = Path('/content/drive/MyDrive/style_matching_runtime')
OLD_ROOT = Path('/content/drive/MyDrive/stylematch_v1')
RUNTIME.mkdir(parents=True, exist_ok=True)
assert REPO.exists(), f'Missing repo: {REPO}'
%cd /content/drive/MyDrive/style_matching

In [ ]:
!git pull origin main
!git rev-parse --short HEAD

In [ ]:
if OLD_ROOT.exists():
    !python scripts/merge_existing_corpus.py --source-root "{OLD_ROOT}" --corpus literary
else:
    print('old root not found; skipping', OLD_ROOT)

MANIFEST = REPO / 'data/source_registry/source_manifest.csv'
if MANIFEST.exists():
    !python scripts/import_source_manifest.py "{MANIFEST}" --dry-run
    !python scripts/import_source_manifest.py "{MANIFEST}" --append
else:
    print('repo manifest not found; skipping', MANIFEST)

In [ ]:
chunks_with_text = RUNTIME / 'data/literary/meta/literary_all_chunks_with_text.parquet'
coverage_report = RUNTIME / 'data/literary/meta/literary_all_coverage_audit.json'

!python scripts/build_chunk_parquet_from_sources.py --corpus literary --language en --min-words 75 --max-words 150 --output "{chunks_with_text}" --coverage-output "{coverage_report}"

In [ ]:
heldout_split = RUNTIME / 'data/literary/meta/literary_all_source_heldout_splits.parquet'
heldout_report = RUNTIME / 'data/literary/meta/literary_all_source_heldout_report.json'
robust_out = RUNTIME / 'results/literary_all_source_heldout_style_robust'
baseline_out = RUNTIME / 'results/literary_all_source_heldout_baseline'

!python scripts/build_chunks_with_text.py --corpus literary --language en --output "{chunks_with_text}"
!python scripts/make_source_heldout_splits.py --input "{chunks_with_text}" --output "{heldout_split}" --report "{heldout_report}"
!python scripts/style_robust_baseline.py --input "{heldout_split}" --out-dir "{robust_out}"
!python scripts/literary_baseline.py --input "{heldout_split}" --out-dir "{baseline_out}"

In [ ]:
import json

robust_metrics = json.loads((robust_out / 'style_robust_metrics.json').read_text())
baseline_metrics = json.loads((baseline_out / 'literary_baseline_metrics.json').read_text())
robust_metrics['models'], baseline_metrics['models']